In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from link_prediction.complexity_analysis import (
    run_complexity_analysis,
)
from link_prediction.config import (
    RESULTS_DIR,
)
from link_prediction.runtime_analysis import (
    run_runtime_analysis,
)
from link_prediction.statistical_analysis import (
    load_benchmark_fold_metrics,
)

In [ ]:
benchmark_name = "standard"

figure_directory = (
    RESULTS_DIR
    / "figures"
    / benchmark_name
)

figure_directory.mkdir(
    parents=True,
    exist_ok=True,
)

fold_metrics = (
    load_benchmark_fold_metrics(
        benchmark_name=
            benchmark_name,
    )
)

runtime_results = run_runtime_analysis(
    fold_metrics=
        fold_metrics,
    benchmark_name=
        benchmark_name,
)

complexity_table = (
    run_complexity_analysis()
)

family_folds = runtime_results[
    "family_folds"
]

network_runtime = runtime_results[
    "network_runtime"
]

family_runtime = runtime_results[
    "family_runtime"
]

In [ ]:
family_runtime[
    [
        "family",
        "analysis_family",
        "network_count",
        "method_count",
        "mean_candidate_count",
        "mean_scoring_seconds",
        "median_scoring_seconds",
        "mean_candidates_per_second",
    ]
].sort_values(
    [
        "mean_scoring_seconds",
        "family",
    ]
).reset_index(
    drop=True
)

In [ ]:
runtime_plot = (
    family_runtime
    .sort_values(
        [
            "mean_scoring_seconds",
            "family",
        ]
    )
)

figure, axis = plt.subplots(
    figsize=(
        9,
        6,
    )
)

axis.barh(
    runtime_plot[
        "family"
    ],
    runtime_plot[
        "mean_scoring_seconds"
    ],
    color="#4472C4",
)

axis.set_xscale(
    "log"
)

axis.set_xlabel(
    "Mean family-batch scoring time in seconds"
)

axis.set_ylabel(
    "Execution family"
)

axis.grid(
    axis="x",
    alpha=0.25,
)

figure.tight_layout()

figure.savefig(
    figure_directory
    / "family_runtime.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "family_runtime.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
runtime_matrix = (
    network_runtime
    .pivot(
        index="family",
        columns="network",
        values="mean_scoring_seconds",
    )
    .sort_index()
)

figure, axis = plt.subplots(
    figsize=(
        12,
        6,
    )
)

image = axis.imshow(
    runtime_matrix.to_numpy(),
    aspect="auto",
    cmap="YlOrRd",
    norm=LogNorm(
        vmin=float(
            runtime_matrix.min().min()
        ),
        vmax=float(
            runtime_matrix.max().max()
        ),
    ),
)

axis.set_xticks(
    range(
        len(
            runtime_matrix.columns
        )
    )
)

axis.set_xticklabels(
    runtime_matrix.columns,
    rotation=45,
    ha="right",
)

axis.set_yticks(
    range(
        len(
            runtime_matrix.index
        )
    )
)

axis.set_yticklabels(
    runtime_matrix.index,
)

axis.set_xlabel(
    "Network"
)

axis.set_ylabel(
    "Execution family"
)

colorbar = figure.colorbar(
    image,
    ax=axis,
)

colorbar.set_label(
    "Mean family-batch scoring time in seconds"
)

figure.tight_layout()

figure.savefig(
    figure_directory
    / "network_family_runtime_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "network_family_runtime_heatmap.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
complexity_table[
    [
        "method",
        "family",
        "analysis_family",
        "complexity",
        "complexity_latex",
    ]
].sort_values(
    [
        "analysis_family",
        "family",
        "method",
    ]
).reset_index(
    drop=True
)